# pretrained HuggingFace transformers

In [1]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")

print(tokenizer.tokenize('ai can be misinterpreted as artificial intelligence'))

c:\Users\aseva\Desktop\MyEDU\YaDLE\YaDLE_project_vscode_stream_2\.venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


['ai', 'can', 'be', 'mis', '##int', '##er', '##pre', '##ted', 'as', 'artificial', 'intelligence']


In [2]:
tokenizer.tokenize('I am fully fucked up and leave for Nice')

['i', 'am', 'fully', 'fucked', 'up', 'and', 'leave', 'for', 'nice']

In [3]:
tokenizer.tokenize('je suis completement vide et pars pour Nice')

['je',
 'sui',
 '##s',
 'complete',
 '##ment',
 'vi',
 '##de',
 'et',
 'par',
 '##s',
 'pour',
 'nice']

In [4]:
tokenizer.tokenize('я заебался и поехал в Ниццу')

['я',
 'з',
 '##а',
 '##е',
 '##б',
 '##а',
 '##л',
 '##с',
 '##я',
 'и',
 'п',
 '##о',
 '##е',
 '##х',
 '##а',
 '##л',
 'в',
 'н',
 '##и',
 '##ц',
 '##ц',
 '##у']

#### ВЫВОД: обучен только на английский корпус токенов

# top frequent tokens - class `Counter` from `collections`

In [6]:
from collections import Counter

Counter(tokenizer.tokenize('я заебался и поехал в Ниццу')).most_common(3)

[('##а', 3), ('##е', 2), ('##л', 2)]

# hand made padding in torch `Dataset`

In [12]:
# импортируем нужные библиотеки
import torch
from torch.utils.data import Dataset, DataLoader

# для совместимости с другими методами класс нашего датасета наследуем от класса Dataset из PyTorch
class PaddedDataset(Dataset):
    # в конструкторе просто сохраняем все данные 
    def __init__(self, texts, labels, max_len):
        self.texts = texts
        self.labels = labels
        self.max_len = max_len

    # метод __len__ возвращает количество объектов в датасете
    def __len__(self):
        return len(self.texts)

    # метод __getitem__ возвращает элемент датасета с индексом idx
    def __getitem__(self, idx):
        # получаем текст и его класс по индексу
        text = self.texts[idx]
        label = self.labels[idx]
        
        # вручную реализуем padding и masking
        padded = text + [0] * (self.max_len - len(text))
        mask = [1] * len(text) + [0] * (self.max_len - len(text))

        # возвращаем текст после padding'а, маску и класс
        return {
            'texts': torch.tensor(padded, dtype=torch.long),
            'masks': torch.tensor(mask, dtype=torch.long),
            'labels': torch.tensor(label, dtype=torch.long)
        }

texts = [[5, 9, 12], [2, 45, 23, 11], [12]]
labels = [1, 0, 1]
max_len = 5

# создаем объект датасета
dataset = PaddedDataset(texts, labels, max_len)

# используем готовый DataLoader из PyTorch
dataloader = DataLoader(dataset, batch_size=2, shuffle=False)

# выводим батчи, который формируются в даталоадере
counter = 1
for batch in dataloader:
    print('\n------------начало батча', str(counter)+'--------------')
    print('\n texts:\n', batch['texts'])
    print('\n masks:\n', batch['masks'])
    print('\n labels:\n', batch['labels'])
    counter += 1



------------начало батча 1--------------

 texts:
 tensor([[ 5,  9, 12,  0,  0],
        [ 2, 45, 23, 11,  0]])

 masks:
 tensor([[1, 1, 1, 0, 0],
        [1, 1, 1, 1, 0]])

 labels:
 tensor([1, 0])

------------начало батча 2--------------

 texts:
 tensor([[12,  0,  0,  0,  0]])

 masks:
 tensor([[1, 0, 0, 0, 0]])

 labels:
 tensor([1])


# `collate_fn` made padding in torch `DataLoader`
Представьте, что в батч попали только короткие последовательности. Тогда нет смысла дополнять их всех до длинных — можно просто сделать `padding` до самой длинной последовательности в батче и сэкономить ресурсы. Это главный плюс этого подхода к реализации `padding` в сравнении с уровнем всего датасета.

In [14]:
# имортируем нужные библиотеки
#import torch
#from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence

# создаем датасет, наследуясь от класса Dataset из PyTorch
class RawDataset(Dataset):
    # в конструкторе просто сохраняем тексты и классы
    def __init__(self, texts, labels, max_len):
        self.texts = texts
        self.labels = labels
        self.max_len = max_len

    # возвращаем размер датасета (кол-во текстов)
    def __len__(self):
        return len(self.texts)
    
    def __getitem__(self, idx):
        # возвращаем текст и его класс
        # для текста ограничиваем длину
        # не делаем никаких доп. преобразований как padding и masking
        return {
            'text': torch.tensor(self.texts[idx][:self.max_len], dtype=torch.long),
            'label': torch.tensor(self.labels[idx], dtype=torch.long)
        }

def collate_fn(batch):
    # список текстов и классов из батча
    texts = [item['text'] for item in batch]
    labels = torch.stack([item['label'] for item in batch])

    # дополняем тексты в батче padding'ом
    padded_texts = pad_sequence(texts, batch_first=True, padding_value=0)

    # считаем маски
    masks = (padded_texts != 0).long()

    # возвращаем преобразованный батч
    return {
        'texts': padded_texts,
        'masks': masks,
        'labels': labels
    }

#texts = [[5, 9, 12], [2, 45, 23, 11], [12]]
#labels = [1, 0, 1]
#max_len = 5

# создаем объект датасета
dataset = RawDataset(texts, labels, max_len)

# пользуемся готовым даталоадером из PyTorch, но с кастомной функцией collate_fn
dataloader = DataLoader(dataset, batch_size=2, collate_fn=collate_fn)

counter = 1
for batch in dataloader:
    print('\n------------начало батча', str(counter)+'--------------')
    print('\n texts:\n', batch['texts'])
    print('\n masks:\n', batch['masks'])
    print('\n labels:\n', batch['labels'])
    counter += 1


------------начало батча 1--------------

 texts:
 tensor([[ 5,  9, 12,  0],
        [ 2, 45, 23, 11]])

 masks:
 tensor([[1, 1, 1, 0],
        [1, 1, 1, 1]])

 labels:
 tensor([1, 0])

------------начало батча 2--------------

 texts:
 tensor([[12]])

 masks:
 tensor([[1]])

 labels:
 tensor([1])


# NB!-s:

In [ ]:
[0]*3, [0]*(3-6)

[]

In [4]:
src_list = [
    ['1'],
    ['1','2'],
    ['1','2','3']
]

[el for el in src_list]

[['1'], ['1', '2'], ['1', '2', '3']]